In [0]:
WITH
-- group household
gh_agg as (
  select orgkey, max(group_id) as group_id
  from lakehouse_prod.bronze_cbs.grouphousehold
  group by orgkey
),
-- group names
cg_agg as (
  select groupid, max(groupname) as groupname
  from lakehouse_prod.bronze_cbs.cifgroups
  group by groupid
),
-- facility extension table
lnm_agg as (
  select *
  from lakehouse_prod.silver_cbs.c_lnm_ext
  where bank_id = '01'
),
-- department description
lrct_dept_agg as (
  select ref_code, max(ref_desc) as ref_desc
  from lakehouse_prod.silver_cbs.c_lrct
  where ref_rec_type = 'LDEPT'
  group by ref_code
),
-- facility type description
lrct_fact_agg as (
  select ref_code, max(ref_desc) as ref_desc
  from lakehouse_prod.silver_cbs.c_lrct
  where ref_rec_type = 'LFACT'
  group by ref_code
),
-- limit category description
lrct_4b_agg as (
  select ref_code, max(ref_desc) as ref_desc
  from lakehouse_prod.bronze_cbs.reference_code_table
  where ref_rec_type = '4B'
  group by ref_code
),
-- marketing flag description
lrct_mkt_agg as (
  select ref_code, max(ref_desc) as ref_desc
  from lakehouse_prod.bronze_cbs.reference_code_table
  where ref_rec_type = 'IT'
  group by ref_code
),
-- OAU region
oau_agg as (
  select countrycode, max(oau) as oau
  from lakehouse_prod.silver_cbs.cust_oau
  group by countrycode
),
-- first disbursement date fallback
disb_agg as (
  select parent_id, min(min_disbursement_date) as min_disbursement_date
  from lakehouse_prod.silver_cbs.ifrs9_min_disb_non_fund
  group by parent_id
),
-- reporting sector
ssmap_agg as (
  select subsector, max(subsectormap) as subsectormap
  from lakehouse_prod.silver_cbs.cust_subsectormap
  group by subsector
),
-- sub-sector localetext
subsector_agg as (
  select
    lnm_sub.limit_prefix,
    lnm_sub.limit_suffix,
    max(cl.localetext) as localetext
  from lakehouse_prod.silver_cbs.c_lnm_ext lnm_sub
  join lakehouse_prod.bronze_cbs.categories ct
    on ct.categorytype = 'SUB_SECTOR_CODE'
    and ct.bank_id = '01'
    and upper(ct.value) = lnm_sub.sub_sector_being_financed
    and coalesce(ct.del_flg,'N') = 'N'
  join lakehouse_prod.bronze_cbs.category_lang cl
    on cl.categoryid = ct.categoryid
    and cl.bank_id = ct.bank_id
    and cl.localecode = 'en_US'
  group by lnm_sub.limit_prefix, lnm_sub.limit_suffix
),
-- contingent liability adjustments
gcaf_agg as (
  select
    parent_id,
    max(contingentliabamtinusd1) as contingentliabamtinusd1,
    max(contingentliabamt1) as contingentliabamt1
  from lakehouse_prod.silver_cbs.view_fac_master_gcaf
  group by parent_id
),
-- loan general details
lnbgen_agg as (
  select
    llt_inner.limit_prefix,
    llt_inner.limit_suffix,
    max(ln.limit_classifier) as limit_classifier,
    max(ln.limit_ctrl_ctr) as limit_ctrl_ctr
  from lakehouse_prod.bronze_cbs.ln_general_details_table ln
  join lakehouse_prod.bronze_cbs.limit_liab_table llt_inner
    on ln.limit_b2kid = llt_inner.limit_b2kid
  group by llt_inner.limit_prefix, llt_inner.limit_suffix
),
-- corporate RM IDs
corp_agg as (
  select corp_key, max(tertiaryrmlogin_id) as tertiaryrmlogin_id, max(secondrmlogin_id) as secondrmlogin_id
  from lakehouse_prod.bronze_cbs.corporate
  group by corp_key
),
-- stat date for currency conversion
sgct_agg as (
  select max(date_sub(db_stat_date, 1)) as stat_date
  from lakehouse_prod.bronze_cbs.sol_group_control_table
),

-- core facility dataset
facility_base as (
  select
    -- fin.lim_exp_date,
    fin.orig_lim_exp_date,
    gh.group_id,
    cg.groupname,
    llt12.availability_end_date,
    llt12.lim_contract_date,
    lnm.main_dept as main_dept_code,
    lrct_dept.ref_desc as main_dept_code_desc,
    fin.salesforce_facility_id,
    fin.cust_name,
    fin.limit_desc,
    fin.risk_country_code,
    fin.sales_sol_id,
    oau.oau as oau_region,
    fin.type_of_dept,
    fin.trade_direct,
    fin.primaryrmlogin_id,
    fin.primary_dept_rm,
    fin.lim_sanct_date,
    case when fin.first_disb_dt is null then disb.min_disbursement_date else fin.first_disb_dt end as firstdisbursementdate,
    lnm.original_credit_grade as originalfacilitygrade,
    fin.loan_grade,
    case
      when fin.loan_grade = '01' then 'VERY LOW RISK'
      when fin.loan_grade in ('02','03') then 'LOW RISK'
      when fin.loan_grade in ('04','05','06','07','08') then 'SATISFACTORY RISK'
      when fin.loan_grade in ('09','10') then 'MODERATE RISK'
      when fin.loan_grade = '11' then 'WATCH LIST RISK'
      when fin.loan_grade = '12' then 'SUB-STANDARD RISK'
      when fin.loan_grade = '13' then 'DOUBTFUL AND BAD DEBT RISK'
      when fin.loan_grade = '14' then 'DEFAULT/LOSS RISK'
    end as riskcategory,
    fin.status_code,
    fin.sector_being_financed,
    cl_sub.localetext as subsector,
    ssmap.subsectormap as reportingsector,
    lnm.int_tbl_code as interesttablecode,
    lnm.acct_margin_pcnt as accountmargin,
    fin.program_loan_type,
    lakehouse_prod.custom.get_new_currency_val(
      lnm.total_facility_amt,
      lnm.total_facility_crncy,
      'USD',
      lakehouse_prod.custom.get_bod_date('01')
    ) as totalfacilityamountusd,
    lnm.total_facility_amt as totalfacilityamtorgccy,
    lnm.total_facility_crncy as totalfacilitycurrency,
    lrct_fact.ref_desc as facilitytype,
    fin.legalentity_type,
    case
      when fin.legalentity_type in ('LIMITED LIABILITY COMPANY','MULTILATERAL','NON - GOVERNMENT FINANCIAL INSTITUTION','PRIVATE COMPANY LIMITED BY SHARES (LTD)','PUBLIC LIMITED COMPANY (PLC)') then 'PRIVATE'
      when fin.legalentity_type in ('GOVERNMENT FINANCIAL INSTITUTION') then 'PUBLIC'
      when fin.legalentity_type in ('MINISTRY', 'PARASTATAL','CENTRAL BANK') then 'SOVEREIGN'
    end as borrowertype,
    fin.orig_sanct_lim,
    lakehouse_prod.custom.get_new_currency_val(
      fin.orig_sanct_lim,
      'USD',
      llt12.crncy_code,
      lakehouse_prod.custom.get_bod_date('01')
    ) as approvedlimitorgccy,
    llt12.committed_flg as committed,
    fin.undrawn_lim,
    lakehouse_prod.custom.get_new_currency_val(
      fin.undrawn_lim,
      'USD',
      llt12.crncy_code,
      lakehouse_prod.custom.get_bod_date('01')
    ) as undrawnlimitorgccy,
    fin.funded_out_bal,
    vloanfac.total_exp_usd,
    fin.funded_out_bal_org,
    case when llt12.crncy_code = 'USD' then vloanfac.total_exp_usd else coalesce(lakehouse_prod.custom.get_new_currency_val(vloanfac.total_exp_usd,'USD',llt12.crncy_code,sgct.stat_date),0) end as total_exp,
    fin.cont_out_bal - coalesce(gcaf.contingentliabamtinusd1,0) as cont_out_bal,
    fin.cont_out_bal_org - coalesce(gcaf.contingentliabamt1,0) as cont_out_bal_org,
    fin.gross_exp,
    fin.oper_exposure,
    fin.apportioned_value,
    fin.adj_coll_value,
    case when sign(fin.net_exposure) = -1 then 0 else fin.net_exposure end as netexposureaftermitiga,
    case
      when lnbgen.limit_classifier = 'B' then 'Bilateral'
      when lnbgen.limit_classifier = 'A' then 'Agent and participant'
      when lnbgen.limit_classifier = 'S' then 'Syndicated Participation'
      else 'Club Deal'
    end as limit_classifier,
    lnbgen.limit_ctrl_ctr as oldfacility,
    lnm.restructured_facility,
    emp_lore.emp_name as lorecqasmanager,
    emp_baop.emp_name as baopmanager,
    llt12.modify_delete_reason_code,
    lrct_mkt.ref_desc as marketing_flag,
    llt12.limit_category,
    lrct_4b.ref_desc as limit_category_desc

  from lakehouse_prod.silver_cbs.main_view_facility_mast fin
  inner join lakehouse_prod.bronze_cbs.limit_liab_table llt12
    on fin.salesforce_facility_id = llt12.limit_prefix || '/' || llt12.limit_suffix
    and llt12.del_flg != 'Y'
  left join lakehouse_prod.silver_cbs.facility_mast_loan_exp_view vloanfac
    on vloanfac.parent_limit = fin.salesforce_facility_id
  left join lnm_agg lnm
    on lnm.limit_prefix || '/' || lnm.limit_suffix = fin.salesforce_facility_id
  left join lrct_dept_agg lrct_dept
    on lrct_dept.ref_code = lnm.main_dept
  left join lrct_fact_agg lrct_fact
    on lrct_fact.ref_code = lnm.type_of_facility
  left join lrct_4b_agg lrct_4b
    on lrct_4b.ref_code = llt12.limit_category
  left join lrct_mkt_agg lrct_mkt
    on lrct_mkt.ref_code = llt12.modify_delete_reason_code
  left join oau_agg oau
    on oau.countrycode = fin.country_code
  left join disb_agg disb
    on disb.parent_id = fin.salesforce_facility_id
  left join ssmap_agg ssmap
    on ssmap.subsector = fin.sub_sector_being_financed
  left join subsector_agg cl_sub
    on cl_sub.limit_prefix || '/' || cl_sub.limit_suffix = fin.salesforce_facility_id
  left join gcaf_agg gcaf
    on gcaf.parent_id = fin.salesforce_facility_id
  left join lnbgen_agg lnbgen
    on lnbgen.limit_prefix || '/' || lnbgen.limit_suffix = fin.salesforce_facility_id
  left join gh_agg gh
    on gh.orgkey = fin.cust_id
  left join cg_agg cg
    on cg.groupid = gh.group_id
  left join corp_agg corp
    on corp.corp_key = fin.cust_id
  left join lakehouse_prod.bronze_cbs.gen_emp_table emp_lore
    on emp_lore.emp_id = corp.tertiaryrmlogin_id
  left join lakehouse_prod.bronze_cbs.gen_emp_table emp_baop
    on emp_baop.emp_id = corp.secondrmlogin_id
  left join sgct_agg sgct
    on 1=1

  where limit_state != 'C'
    and not exists (
      select 1 from lakehouse_prod.silver_cbs.view_cot_chrge_off_new cot
      where cot.parent_id = fin.salesforce_facility_id
    )
)

-- final output layer
select
  group_id,
  groupname,
  salesforce_facility_id as facility_id,
  cust_name as customer_name,
  limit_desc as facility_description,
  risk_country_code as country,
  sales_sol_id as region,
  oau_region,
  main_dept_code_desc as department,
  trade_direct as direction_of_trade,
  primaryrmlogin_id as client_relationship_manager,
  primary_dept_rm as product_manager,
  date_format(lim_sanct_date, 'MM/dd/yyyy') as facility_approval_date,
  lim_contract_date as contract_sign_date,
  date_format(firstdisbursementdate, 'MM/dd/yyyy') as first_disbursement_date,
  originalfacilitygrade as original_facility_grade,
  loan_grade as facility_grade,
  riskcategory as risk_category,
  status_code as facility_status,
  availability_end_date as `availability_period-end_date`,
  --(case when lim_exp_date > orig_lim_exp_date then date_format(lim_exp_date, 'MM/dd/yyyy') else date_format(orig_lim_exp_date, 'MM/dd/yyyy') end) as facility_expiry_date,
  sector_being_financed as sector,
  subsector as `sub-sector`,
  reportingsector as reporting_sector,
  interesttablecode as interest_table_code,
  accountmargin as account_margin_pct,
  program_loan_type as program,
  totalfacilityamountusd as total_facility_amt_usd,
  totalfacilityamtorgccy as total_facility_amt_org_ccy,
  totalfacilitycurrency as total_facility_currency,
  facilitytype as facility_type,
  legalentity_type as beneficiary_type,
  borrowertype as borrower_type,
  orig_sanct_lim as approved_limit,
  approvedlimitorgccy as approved_limit_org_ccy,
  committed,
  coalesce(undrawn_lim,0) as undrawn_limit,
  coalesce(undrawnlimitorgccy,0) as undrawn_limit_orginal_ccy,
  coalesce(funded_out_bal,0) as principal_bal_funded,
  coalesce(total_exp_usd,0) as total_exposure_funded,
  coalesce(funded_out_bal_org,0) as principal_bal_funded_org_ccy,
  coalesce(total_exp,0) as total_exposure_funded_org_ccy,
  coalesce(cont_out_bal,0) as outstanding_bal_contig,
  coalesce(cont_out_bal_org,0) as outstand_bal_contig_org_ccy,
  coalesce((coalesce(undrawn_lim,0) + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)),0) as gross_exposure,
  coalesce(case when status_code = 'NON-OPERATIONAL' then 0 else (case when committed = 'N' then 0 else coalesce(undrawn_lim,0) end + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)) end,0) as operational_exposure,
  coalesce(apportioned_value,0) as collateral_value_gross,
  coalesce(adj_coll_value,0) as collateral_value_adjusted,
  coalesce(greatest(case when status_code = 'NON-OPERATIONAL' then 0 else (case when committed = 'N' then 0 else coalesce(undrawn_lim,0) end + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)) end - coalesce(adj_coll_value,0),0,0),0) as net_exposure,
  limit_classifier,
  oldfacility,
  restructured_facility,
  lorecqasmanager as lore_cqas_manager,
  baopmanager,
  marketing_flag,
  limit_category,
  limit_category_desc
from facility_base
where coalesce((coalesce(undrawn_lim,0) + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)),0) != 0